# SQL 기초 — 관계형 데이터베이스와 기본 조회  (SQLite 노트북)
원본 파일: `w5_SQL_기초_관계형_데이터베이스와_기본_조회.sql` · SQL 셀 38개

## 사용법
1. **위에서부터 순서대로** 실행하세요. 앞쪽 셀이 테이블(`customers`, `orders` 등)을 만듭니다. 데이터 생성 셀은 `DROP TABLE IF EXISTS`로 시작해서 다시 실행해도 안전합니다.
2. 연습 문제 셀은 주석 아래에 SQL을 쓰고 실행합니다. 원본 파일에 이미 풀이가 들어 있던 셀은 그대로 가져왔습니다.
3. 준비물: Python 3, `pandas` (`pip install pandas`), VS Code의 Python·Jupyter 확장. `sqlite3`는 파이썬에 내장되어 있어서 설치가 필요 없습니다.

## 원본(BigQuery/DuckDB)에서 자동 변환한 내용
- 테이블 경로 `` `aiffel-515805.modumall.x` ``, `project_name.dataset_name.x` → `x`
- `CREATE OR REPLACE TABLE` → `DROP TABLE IF EXISTS` + `CREATE TABLE`, 열 타입 `STRING`/`INT64`/`DECIMAL`/`DATE` → `TEXT`/`INTEGER`/`REAL`/`TEXT`
- 날짜는 SQLite에 별도 타입이 없어서 `'YYYY-MM-DD'` 텍스트로 저장합니다.

### 셀별 추가 변환
- **C5**: DESCRIBE → PRAGMA table_info
- **C6**: DESCRIBE → PRAGMA table_info

> 검증: 원본 SQL을 DuckDB에서, 변환본을 SQLite에서 각각 실행해 셀별 결과를 비교했습니다. 값이 다른 셀은 없었습니다. (정렬 기준이 없는 쿼리는 행 순서가, 열 이름을 직접 안 붙인 `count(*)` 같은 열은 자동 이름이 엔진마다 다를 수 있습니다.)

In [9]:
import re, sqlite3
import pandas as pd
try:
    from IPython.display import display
except ImportError:          # 일반 파이썬에서 실행할 때
    display = print

con = sqlite3.connect(":memory:")   # 메모리 DB: 노트북(커널)을 끄면 사라집니다

def sql(text):
    """SQL을 실행합니다. SELECT류는 표로 보여주고, CREATE/INSERT 등은 실행만 합니다."""
    stmts, buf = [], ""
    for line in text.splitlines(keepends=True):
        buf += line
        if sqlite3.complete_statement(buf):      # ; 로 끝나는 문장 단위로 분리
            stmts.append(buf.strip()); buf = ""
    if buf.strip():
        stmts.append(buf.strip())
    ran = shown = 0
    for st in stmts:
        body = re.sub(r"--[^\n]*|/\*.*?\*/", "", st, flags=re.S).strip().rstrip(";").strip()
        if not body:
            continue                              # 주석만 있는 조각은 건너뜀
        ran += 1
        if body.split(None, 1)[0].upper() in ("SELECT", "WITH", "PRAGMA", "VALUES"):
            display(pd.read_sql_query(st, con)); shown += 1
        else:
            con.execute(st)
    con.commit()
    if ran == 0:
        print("(아직 작성한 SQL이 없습니다. 주석 아래에 쿼리를 작성한 뒤 다시 실행하세요.)")
    elif shown == 0:
        print("실행 완료")

print("SQLite 버전:", sqlite3.sqlite_version)

SQLite 버전: 3.53.4


## 1. 두 환경의 문법 차이 — 한눈에 보기

> **[C1] 건너뜀** — 원본의 DuckDB/BigQuery용 이름 공간 준비 셀입니다. SQLite는 필요 없습니다.

## 2. 모두몰 데이터 만들기

In [10]:
sql(r"""
-- [C2]
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
  customer_id TEXT, /* 고객번호 (예: C001) */
  name TEXT, /* 이름 */
  country TEXT, /* 국가 */
  signup_date TEXT, /* 가입일 */
  grade TEXT /* 등급 (Gold/Silver/Bronze) */
);

INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');
""")

실행 완료


In [11]:
sql(r"""
-- [C3]
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_id TEXT, /* 주문번호 */
  customer_id TEXT, /* 고객번호 (customers와 연결) */
  order_date TEXT, /* 주문일 */
  status TEXT, /* 상태 (Placed/Paid/Shipped/Cancelled/Returned) */
  amount REAL /* 결제 금액 (취소 시 비어 있을 수 있음) */
);

INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")

실행 완료


In [12]:
sql(r"""
-- [C4] customers 테이블에 데이터가 들어갔는지 결과 중 5행을 확인합니다.
SELECT
  *
FROM customers
LIMIT 5;
""")

,customer_id,name,country,signup_date,grade
0,C001,김민준,Korea,2023-01-05,Gold
1,C002,이서연,Korea,2023-02-11,Silver
2,C003,박도윤,Japan,2023-02-20,Bronze
3,C004,최지우,USA,2023-03-03,Gold
4,C005,정하준,Korea,2023-03-15,Silver


## 3. 스키마 — 테이블의 설계도

In [13]:
sql(r"""
-- [C5] LMS(DuckDB): customers 테이블의 구조를 확인합니다.
PRAGMA table_info(customers);

-- BigQuery에서는 위 문장 대신 아래 주석의 SELECT 문을 사용합니다.
-- SELECT column_name, data_type
-- FROM INFORMATION_SCHEMA.COLUMNS
-- WHERE table_name = 'customers'
-- ORDER BY ordinal_position;
""")

,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,TEXT,0,None,0
1,1,name,TEXT,0,None,0
2,2,country,TEXT,0,None,0
3,3,signup_date,TEXT,0,None,0
4,4,grade,TEXT,0,None,0


## 4. ⌨️ 백문이 불여일타 (1)

In [14]:
sql(r"""
-- [C6] ⌨️ 백문이 불여일타 (1) — orders 테이블 스키마 확인
PRAGMA table_info(orders);
""")

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,order_date,TEXT,0,None,0
3,3,status,TEXT,0,None,0
4,4,amount,REAL,0,None,0


### 5. ▶️ **코드 실행하기 · 코드 셀 7 [C7]**

In [15]:
sql(r"""
-- [C7]
SELECT
  name,
  country,
  grade
FROM customers;
""")

,name,country,grade
0,김민준,Korea,Gold
1,이서연,Korea,Silver
2,박도윤,Japan,Bronze
3,최지우,USA,Gold
4,정하준,Korea,Silver
5,강서윤,Korea,Bronze
6,조은우,Japan,Silver
7,윤지호,USA,Gold
8,임하은,Korea,Bronze
9,한예준,Korea,Silver


## 6. ⌨️ 백문이 불여일타 (2)

In [16]:
sql(r"""
-- [C8] ⌨️ 백문이 불여일타 (2) — orders에서 세 열만 골라 조회

SELECT
  order_id,
  order_date,
  amount
FROM orders;
""")

,order_id,order_date,amount
0,O0001,2023-09-02,125000.0
1,O0002,2023-09-05,89000.0
2,O0003,2023-09-11,45000.0
3,O0004,2023-09-15,230000.0
4,O0005,2023-09-20,NaN
5,O0006,2023-09-25,67000.0
6,O0007,2023-10-01,158000.0
7,O0008,2023-10-04,32000.0
8,O0009,2023-10-12,410000.0
9,O0010,2023-10-19,99000.0


## 7. ⌨️ 백문이 불여일타 (3)

In [17]:
sql(r"""
-- [C9] ⌨️ 백문이 불여일타 (3) — customers 열 순서 바꿔 조회

SELECT
  grade,
  name,
  country
FROM customers;
""")

,grade,name,country
0,Gold,김민준,Korea
1,Silver,이서연,Korea
2,Bronze,박도윤,Japan
3,Gold,최지우,USA
4,Silver,정하준,Korea
5,Bronze,강서윤,Korea
6,Silver,조은우,Japan
7,Gold,윤지호,USA
8,Bronze,임하은,Korea
9,Silver,한예준,Korea


### 8. ▶️ **코드 실행하기 · 코드 셀 10 [C10]**

In [18]:
sql(r"""
-- [C10]
SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  status = 'Paid';
""")

,order_id,status,amount
0,O0001,Paid,125000.0
1,O0004,Paid,230000.0
2,O0007,Paid,158000.0
3,O0010,Paid,99000.0
4,O0011,Paid,76000.0
5,O0014,Paid,88000.0
6,O0018,Paid,320000.0
7,O0019,Paid,47000.0
8,O0022,Paid,134000.0
9,O0024,Paid,268000.0


### 9. ▶️ **코드 실행하기 · 코드 셀 11 [C11]**

In [19]:
sql(r"""
-- [C11]
SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  amount >= 100000;
""")

,order_id,status,amount
0,O0001,Paid,125000.0
1,O0004,Paid,230000.0
2,O0007,Paid,158000.0
3,O0009,Shipped,410000.0
4,O0013,Shipped,142000.0
5,O0016,Shipped,175000.0
6,O0018,Paid,320000.0
7,O0020,Shipped,215000.0
8,O0022,Paid,134000.0
9,O0024,Paid,268000.0


## 10. ⌨️ 백문이 불여일타 (4)

In [20]:
sql(r"""
-- [C12] ⌨️ 백문이 불여일타 (4) — 취소가 아닌 주문만 조회

SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  status != 'Cancelled';
""")

,order_id,status,amount
0,O0001,Paid,125000.0
1,O0002,Shipped,89000.0
2,O0003,Returned,45000.0
3,O0004,Paid,230000.0
4,O0006,Shipped,67000.0
5,O0007,Paid,158000.0
6,O0008,Placed,32000.0
7,O0009,Shipped,410000.0
8,O0010,Paid,99000.0
9,O0011,Paid,76000.0


## 11. ⌨️ 백문이 불여일타 (5)

In [21]:
sql(r"""
-- [C13] ⌨️ 백문이 불여일타 (5) — 5만원 미만 소액 주문 조회

SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  amount < 50000;
""")

,order_id,status,amount
0,O0003,Returned,45000.0
1,O0008,Placed,32000.0
2,O0019,Paid,47000.0
3,O0021,Placed,38000.0


### 12. ▶️ **코드 실행하기 · 코드 셀 14 [C14]**

In [22]:
sql(r"""
-- [C14]
SELECT
  name,
  country,
  grade
FROM customers
WHERE
  country = 'Korea' AND grade = 'Gold';
""")

,name,country,grade
0,김민준,Korea,Gold
1,권준서,Korea,Gold


### 13. ▶️ **코드 실행하기 · 코드 셀 15 [C15]**

In [23]:
sql(r"""
-- [C15]
SELECT
  name,
  country,
  grade
FROM customers
WHERE
  grade = 'Gold' OR grade = 'Silver';
""")

,name,country,grade
0,김민준,Korea,Gold
1,이서연,Korea,Silver
2,최지우,USA,Gold
3,정하준,Korea,Silver
4,조은우,Japan,Silver
5,윤지호,USA,Gold
6,한예준,Korea,Silver
7,신아린,Japan,Silver
8,권준서,Korea,Gold


### 14. ▶️ **코드 실행하기 · 코드 셀 16 [C16]**

In [24]:
sql(r"""
-- [C16]
SELECT
  order_id,
  amount
FROM orders
WHERE
  amount BETWEEN 50000 AND 100000;
""")

,order_id,amount
0,O0002,89000.0
1,O0006,67000.0
2,O0010,99000.0
3,O0011,76000.0
4,O0014,88000.0
5,O0015,53000.0
6,O0017,61000.0
7,O0023,92000.0
8,O0028,58000.0
9,O0029,73000.0


### 15. ▶️ **코드 실행하기 · 코드 셀 17 [C17]**

In [25]:
sql(r"""
-- [C17]
SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  status IN ('Paid', 'Shipped');
""")

,order_id,status,amount
0,O0001,Paid,125000.0
1,O0002,Shipped,89000.0
2,O0004,Paid,230000.0
3,O0006,Shipped,67000.0
4,O0007,Paid,158000.0
5,O0009,Shipped,410000.0
6,O0010,Paid,99000.0
7,O0011,Paid,76000.0
8,O0013,Shipped,142000.0
9,O0014,Paid,88000.0


## 16. ⌨️ 백문이 불여일타 (6)

In [26]:
sql(r"""
-- [C18] ⌨️ 백문이 불여일타 (6) — 한국·일본 고객 조회

SELECT
  name,
  country,
  grade
FROM customers
WHERE
  country IN ('Korea', 'Japan');
""")

,name,country,grade
0,김민준,Korea,Gold
1,이서연,Korea,Silver
2,박도윤,Japan,Bronze
3,정하준,Korea,Silver
4,강서윤,Korea,Bronze
5,조은우,Japan,Silver
6,임하은,Korea,Bronze
7,한예준,Korea,Silver
8,신아린,Japan,Silver
9,권준서,Korea,Gold


## 17. ⌨️ 백문이 불여일타 (7)

In [27]:
sql(r"""
-- [C19] ⌨️ 백문이 불여일타 (7) — Paid이면서 10만~30만원 주문 조회

SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  status = 'Paid'
  AND amount BETWEEN 100000 AND 300000;
""")

,order_id,status,amount
0,O0001,Paid,125000.0
1,O0004,Paid,230000.0
2,O0007,Paid,158000.0
3,O0022,Paid,134000.0
4,O0024,Paid,268000.0
5,O0026,Paid,119000.0
6,O0030,Paid,187000.0


### 18. ▶️ **코드 실행하기 · 코드 셀 20 [C20]**

In [28]:
sql(r"""
-- [C20]
SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  amount IS NULL;
""")

,order_id,status,amount
0,O0005,Cancelled,None
1,O0012,Cancelled,None
2,O0025,Cancelled,None


### 19. ▶️ **코드 실행하기 · 코드 셀 21 [C21]**

In [29]:
sql(r"""
-- [C21]
SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  amount IS NOT NULL;
""")

,order_id,status,amount
0,O0001,Paid,125000.0
1,O0002,Shipped,89000.0
2,O0003,Returned,45000.0
3,O0004,Paid,230000.0
4,O0006,Shipped,67000.0
5,O0007,Paid,158000.0
6,O0008,Placed,32000.0
7,O0009,Shipped,410000.0
8,O0010,Paid,99000.0
9,O0011,Paid,76000.0


### 20. ▶️ **코드 실행하기 · 코드 셀 22 [C22]**

In [30]:
sql(r"""
-- [C22]
SELECT
  order_id,
  amount,
  amount * 1.1 AS amount_with_tax
FROM orders
WHERE
  amount IS NOT NULL;
""")

,order_id,amount,amount_with_tax
0,O0001,125000.0,137500.0
1,O0002,89000.0,97900.0
2,O0003,45000.0,49500.0
3,O0004,230000.0,253000.0
4,O0006,67000.0,73700.0
5,O0007,158000.0,173800.0
6,O0008,32000.0,35200.0
7,O0009,410000.0,451000.0
8,O0010,99000.0,108900.0
9,O0011,76000.0,83600.0


## 21. ⌨️ 백문이 불여일타 (8)

In [31]:
sql(r"""
-- [C23] ⌨️ 백문이 불여일타 (8) — 국가 정보가 빈 고객 찾기

SELECT
  customer_id,
  name,
  country
FROM customers
WHERE
  country IS NULL;
""")

,customer_id,name,country
0,C011,오시우,None


## 22. ⌨️ 백문이 불여일타 (9)

In [32]:
sql(r"""
-- [C24] ⌨️ 백문이 불여일타 (9) — 금액의 5% 적립 포인트 계산

SELECT
  order_id,
  amount,
  amount * 0.05 AS point
FROM orders
WHERE
  amount IS NOT NULL;
""")

,order_id,amount,point
0,O0001,125000.0,6250.0
1,O0002,89000.0,4450.0
2,O0003,45000.0,2250.0
3,O0004,230000.0,11500.0
4,O0006,67000.0,3350.0
5,O0007,158000.0,7900.0
6,O0008,32000.0,1600.0
7,O0009,410000.0,20500.0
8,O0010,99000.0,4950.0
9,O0011,76000.0,3800.0


## 23. ⌨️ 백문이 불여일타 (10)

In [33]:
sql(r"""
-- [C25] ⌨️ 백문이 불여일타 (10) — 조건 3개를 모두 만족하는 주문 조회

SELECT
  order_id,
  order_date,
  status,
  amount
FROM orders
WHERE
  status IN ('Paid', 'Shipped')
  AND amount >= 100000
  AND order_date >= '2023-11-01';
""")

,order_id,order_date,status,amount
0,O0013,2023-11-02,Shipped,142000.0
1,O0016,2023-11-19,Shipped,175000.0
2,O0018,2023-11-29,Paid,320000.0
3,O0020,2023-12-09,Shipped,215000.0
4,O0022,2023-12-20,Paid,134000.0
5,O0024,2024-01-03,Paid,268000.0
6,O0026,2024-01-15,Paid,119000.0
7,O0027,2024-01-22,Shipped,405000.0
8,O0030,2024-02-11,Paid,187000.0


## 24. 📒 연습 문제 — 조회와 필터링 (10문항)

In [34]:
sql(r"""
-- [C26] 📒 연습 문제 — 조회와 필터링 (1/10)

-- 문제: `status`가 `'Shipped'`인 주문의 `order_id`·`order_date`·`status`를 조회합니다.

SELECT
  order_id,
  order_date,
  status
FROM orders
WHERE
  status = 'Shipped';
""")

,order_id,order_date,status
0,O0002,2023-09-05,Shipped
1,O0006,2023-09-25,Shipped
2,O0009,2023-10-12,Shipped
3,O0013,2023-11-02,Shipped
4,O0016,2023-11-19,Shipped
5,O0020,2023-12-09,Shipped
6,O0023,2023-12-25,Shipped
7,O0027,2024-01-22,Shipped


In [35]:
sql(r"""
-- [C27] 📒 연습 문제 — 조회와 필터링 (2/10)

-- 문제: `status`가 `'Cancelled'`가 아닌 주문을 조회합니다.

SELECT
  *
FROM orders
WHERE
  status != 'Cancelled';
""")

,order_id,customer_id,order_date,status,amount
0,O0001,C001,2023-09-02,Paid,125000.0
1,O0002,C002,2023-09-05,Shipped,89000.0
2,O0003,C001,2023-09-11,Returned,45000.0
3,O0004,C003,2023-09-15,Paid,230000.0
4,O0006,C005,2023-09-25,Shipped,67000.0
5,O0007,C002,2023-10-01,Paid,158000.0
6,O0008,C006,2023-10-04,Placed,32000.0
7,O0009,C007,2023-10-12,Shipped,410000.0
8,O0010,C008,2023-10-19,Paid,99000.0
9,O0011,C001,2023-10-23,Paid,76000.0


In [36]:
sql(r"""
-- [C28] 📒 연습 문제 — 조회와 필터링 (3/10)

-- 문제: `order_date`가 2023-10-01 이후인 주문을 조회합니다. 당일을 포함합니다.

SELECT
  *
FROM orders
WHERE
  order_date >= '2023-10-01';
""")

,order_id,customer_id,order_date,status,amount
0,O0007,C002,2023-10-01,Paid,158000.0
1,O0008,C006,2023-10-04,Placed,32000.0
2,O0009,C007,2023-10-12,Shipped,410000.0
3,O0010,C008,2023-10-19,Paid,99000.0
4,O0011,C001,2023-10-23,Paid,76000.0
5,O0012,C009,2023-10-28,Cancelled,NaN
6,O0013,C010,2023-11-02,Shipped,142000.0
7,O0014,C004,2023-11-08,Paid,88000.0
8,O0015,C011,2023-11-13,Placed,53000.0
9,O0016,C012,2023-11-19,Shipped,175000.0


In [37]:
sql(r"""
-- [C29] 📒 연습 문제 — 조회와 필터링 (4/10)

-- 문제: `amount`가 100,000원 미만인 주문의 `order_id`·`order_date`·`amount`를 조회합니다.

SELECT
  order_id,
  order_date,
  amount
FROM orders
WHERE
  amount < 100000;
""")

,order_id,order_date,amount
0,O0002,2023-09-05,89000.0
1,O0003,2023-09-11,45000.0
2,O0006,2023-09-25,67000.0
3,O0008,2023-10-04,32000.0
4,O0010,2023-10-19,99000.0
5,O0011,2023-10-23,76000.0
6,O0014,2023-11-08,88000.0
7,O0015,2023-11-13,53000.0
8,O0017,2023-11-24,61000.0
9,O0019,2023-12-03,47000.0


In [38]:
sql(r"""
-- [C30] 📒 연습 문제 — 조회와 필터링 (5/10)

-- 문제: `amount`가 150,000원 이하이고 `order_date`가 2023-10-01 이후인 주문을 조회합니다.

SELECT
  *
FROM orders
WHERE
  amount <= 150000
  AND order_date >= '2023-10-01';
""")

,order_id,customer_id,order_date,status,amount
0,O0008,C006,2023-10-04,Placed,32000.0
1,O0010,C008,2023-10-19,Paid,99000.0
2,O0011,C001,2023-10-23,Paid,76000.0
3,O0013,C010,2023-11-02,Shipped,142000.0
4,O0014,C004,2023-11-08,Paid,88000.0
5,O0015,C011,2023-11-13,Placed,53000.0
6,O0017,C002,2023-11-24,Returned,61000.0
7,O0019,C005,2023-12-03,Paid,47000.0
8,O0021,C014,2023-12-14,Placed,38000.0
9,O0022,C001,2023-12-20,Paid,134000.0


In [39]:
sql(r"""
-- [C31] 📒 연습 문제 — 조회와 필터링 (6/10)

-- 문제: `amount`가 100,000원 이하이거나 `order_date`가 2023-11-01 이후인 주문을 조회합니다.

SELECT
  *
FROM orders
WHERE
  amount <= 100000
  OR order_date >= '2023-11-01';
""")

,order_id,customer_id,order_date,status,amount
0,O0002,C002,2023-09-05,Shipped,89000.0
1,O0003,C001,2023-09-11,Returned,45000.0
2,O0006,C005,2023-09-25,Shipped,67000.0
3,O0008,C006,2023-10-04,Placed,32000.0
4,O0010,C008,2023-10-19,Paid,99000.0
5,O0011,C001,2023-10-23,Paid,76000.0
6,O0013,C010,2023-11-02,Shipped,142000.0
7,O0014,C004,2023-11-08,Paid,88000.0
8,O0015,C011,2023-11-13,Placed,53000.0
9,O0016,C012,2023-11-19,Shipped,175000.0


In [40]:
sql(r"""
-- [C32] 📒 연습 문제 — 조회와 필터링 (7/10)

-- 문제: `status`가 `'Placed'` 또는 `'Cancelled'`인 주문을 `IN`으로 조회합니다.

SELECT
  *
FROM orders
WHERE
  status IN ('Placed', 'Cancelled');
""")

,order_id,customer_id,order_date,status,amount
0,O0005,C004,2023-09-20,Cancelled,NaN
1,O0008,C006,2023-10-04,Placed,32000.0
2,O0012,C009,2023-10-28,Cancelled,NaN
3,O0015,C011,2023-11-13,Placed,53000.0
4,O0021,C014,2023-12-14,Placed,38000.0
5,O0025,C010,2024-01-09,Cancelled,NaN


In [41]:
sql(r"""
-- [C33] 📒 연습 문제 — 조회와 필터링 (8/10)

-- 문제: `status`가 `'Placed'`도 `'Cancelled'`도 아닌 주문을 `NOT IN`으로 조회합니다.

SELECT
  *
FROM orders
WHERE
  status NOT IN ('Placed', 'Cancelled');
""")

,order_id,customer_id,order_date,status,amount
0,O0001,C001,2023-09-02,Paid,125000.0
1,O0002,C002,2023-09-05,Shipped,89000.0
2,O0003,C001,2023-09-11,Returned,45000.0
3,O0004,C003,2023-09-15,Paid,230000.0
4,O0006,C005,2023-09-25,Shipped,67000.0
5,O0007,C002,2023-10-01,Paid,158000.0
6,O0009,C007,2023-10-12,Shipped,410000.0
7,O0010,C008,2023-10-19,Paid,99000.0
8,O0011,C001,2023-10-23,Paid,76000.0
9,O0013,C010,2023-11-02,Shipped,142000.0


In [42]:
sql(r"""
-- [C34] 📒 연습 문제 — 조회와 필터링 (9/10)

-- 문제: `amount`가 `NULL`인 주문의 `order_id`·`status`·`amount`를 조회합니다.

SELECT
  order_id,
  status,
  amount
FROM orders
WHERE
  amount IS NULL;
""")

,order_id,status,amount
0,O0005,Cancelled,None
1,O0012,Cancelled,None
2,O0025,Cancelled,None


In [43]:
sql(r"""
-- [C35] 📒 연습 문제 — 조회와 필터링 (10/10)

-- 문제: `order_date`가 2023-10-01부터 2023-10-31까지인 주문을 `BETWEEN`으로 조회합니다.

SELECT
  *
FROM orders
WHERE
  order_date BETWEEN '2023-10-01' AND '2023-10-31';
""")

,order_id,customer_id,order_date,status,amount
0,O0007,C002,2023-10-01,Paid,158000.0
1,O0008,C006,2023-10-04,Placed,32000.0
2,O0009,C007,2023-10-12,Shipped,410000.0
3,O0010,C008,2023-10-19,Paid,99000.0
4,O0011,C001,2023-10-23,Paid,76000.0
5,O0012,C009,2023-10-28,Cancelled,NaN


## 34. 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성

In [44]:
sql(r"""
-- [C36] 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성 (1/3)

SELECT
  order_id,
  order_date,
  status,
  amount
FROM orders
WHERE
  status IN ('Cancelled', 'Returned');
""")

,order_id,order_date,status,amount
0,O0003,2023-09-11,Returned,45000.0
1,O0005,2023-09-20,Cancelled,NaN
2,O0012,2023-10-28,Cancelled,NaN
3,O0017,2023-11-24,Returned,61000.0
4,O0025,2024-01-09,Cancelled,NaN
5,O0029,2024-02-04,Returned,73000.0


## 35. **질문 2.** 금액이 입력된 주문 중 20만원을 초과하는 고액 주문을 조회합니다.

In [45]:
sql(r"""
-- [C37] 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성 (2/3)

SELECT
  order_id,
  order_date,
  status,
  amount
FROM orders
WHERE
  amount IS NOT NULL
  AND amount > 200000;
""")

,order_id,order_date,status,amount
0,O0004,2023-09-15,Paid,230000.0
1,O0009,2023-10-12,Shipped,410000.0
2,O0018,2023-11-29,Paid,320000.0
3,O0020,2023-12-09,Shipped,215000.0
4,O0024,2024-01-03,Paid,268000.0
5,O0027,2024-01-22,Shipped,405000.0


## 36. **질문 3.** 결제 완료(`Paid`) 또는 배송 완료(`Shipped`) 상태인 주문만 추려 시각화에 사용할 데이터를 준비합니다. 상태별 건수 집계는 다음 시간에 `GROUP BY`와 `COUNT`로 다룹니다.

In [46]:
sql(r"""
-- [C38] 🧪 종합 실습 — 조건별 주문 조회 쿼리 작성 (3/3)

SELECT
  order_id,
  order_date,
  status,
  amount
FROM orders
WHERE
  status IN ('Paid', 'Shipped');
""")

,order_id,order_date,status,amount
0,O0001,2023-09-02,Paid,125000.0
1,O0002,2023-09-05,Shipped,89000.0
2,O0004,2023-09-15,Paid,230000.0
3,O0006,2023-09-25,Shipped,67000.0
4,O0007,2023-10-01,Paid,158000.0
5,O0009,2023-10-12,Shipped,410000.0
6,O0010,2023-10-19,Paid,99000.0
7,O0011,2023-10-23,Paid,76000.0
8,O0013,2023-11-02,Shipped,142000.0
9,O0014,2023-11-08,Paid,88000.0


## 37. 코드 퀴즈

In [47]:
sql(r"""
-- [C39] 코드 퀴즈 — 미국이 아니면서 등급이 있는 고객 조회

SELECT
  name,
  country,
  grade
FROM customers
WHERE
  country != 'USA'
  AND grade IS NOT NULL;
""")

,name,country,grade
0,김민준,Korea,Gold
1,이서연,Korea,Silver
2,박도윤,Japan,Bronze
3,정하준,Korea,Silver
4,강서윤,Korea,Bronze
5,조은우,Japan,Silver
6,임하은,Korea,Bronze
7,한예준,Korea,Silver
8,신아린,Japan,Silver
9,권준서,Korea,Gold
